# Multi-agent Matching Experiment

3エージェントの検出シミュレーションを生成し、RoCo-style、Pairwise RRWM、k-wise RRWM の3手法を比較します。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path):
    """現在位置から親方向へ、src/simulation.py を持つ repo root を探す。"""
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "src" / "simulation.py").exists():
            return path
    return None


repo_dir = find_repo_root(Path.cwd())

# VS Code から Colab kernel を使っている場合は /content にいるため、public repo を clone する。
if repo_dir is None and Path("/content").exists():
    repo_dir = Path("/content/Multi-agent-Matching")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/furuyuu/Multi-agent-Matching.git", str(repo_dir)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)

if repo_dir is None or not (repo_dir / "src" / "simulation.py").exists():
    raise FileNotFoundError("Could not find repository root. Check your notebook kernel location.")

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("repo_dir:", Path.cwd())

## Imports

In [ ]:
from src.simulation import (
    count_detection_labels,
    generate_detections,
    generate_true_scene,
)
from src.visualization import (
    plot_detected_scene,
    plot_pairwise_matching_result,
    plot_true_scene,
)
from src.graph_matching import (
    KWISE_PARAMS,
    ROCO_PARAMS,
    RRWM_PARAMS,
    evaluate_kwise_matching_3agents,
    kwise_rrwm_matching_3agents,
    kwise_to_pairwise_matches,
    print_kwise_matches_with_true_ids,
    print_matches_with_true_ids,
    print_method_comparison_table,
    run_pairwise_rrwm_all_pairs,
    run_roco_all_pairs,
)

agent_pairs = [(0, 1), (0, 2), (1, 2)]

## Generate Simulation

In [ ]:
agents_gt, objects_gt = generate_true_scene(
    num_agents=3,
    num_objects=12,
    seed=7,
)

agent_pose_est, detections_by_agent = generate_detections(
    agents_gt,
    objects_gt,
    sensing_range=32.0,
    fov_deg=150.0,
    detection_prob=0.85,
    object_pos_noise_std=0.7,
    object_yaw_noise_std_deg=4.0,
    agent_pos_noise_std=1.0,
    agent_yaw_noise_std_deg=3.0,
    num_outliers_per_agent=2,
    seed=8,
)

for agent_id, (total, inliers, outliers) in count_detection_labels(detections_by_agent).items():
    print(
        f"Agent {agent_id}: detections={total}, "
        f"inliers={inliers}, outliers={outliers}"
    )

In [ ]:
plot_true_scene(agents_gt, objects_gt)
plot_detected_scene(agents_gt, agent_pose_est, detections_by_agent)

## Run Matching Methods

In [ ]:
roco_results = run_roco_all_pairs(
    detections_by_agent,
    agent_pairs,
    ROCO_PARAMS,
)

rrwm_results = run_pairwise_rrwm_all_pairs(
    detections_by_agent,
    agent_pairs,
    RRWM_PARAMS,
)

dets_0 = detections_by_agent[0]
dets_1 = detections_by_agent[1]
dets_2 = detections_by_agent[2]

kwise_matches, kwise_x, kwise_K, kwise_candidates = kwise_rrwm_matching_3agents(
    dets_0,
    dets_1,
    dets_2,
    **KWISE_PARAMS,
)

kwise_eval = evaluate_kwise_matching_3agents(
    dets_0,
    dets_1,
    dets_2,
    kwise_matches,
)

print("k-wise evaluation:", kwise_eval)

## Matching Details

In [ ]:
for i, j in agent_pairs:
    print_matches_with_true_ids(
        f"RoCo-style matching: Agent {i} - Agent {j}",
        detections_by_agent[i],
        detections_by_agent[j],
        roco_results[(i, j)]["matches"],
    )
    print("evaluation:", roco_results[(i, j)]["evaluation"])

for i, j in agent_pairs:
    print_matches_with_true_ids(
        f"Pairwise RRWM matching: Agent {i} - Agent {j}",
        detections_by_agent[i],
        detections_by_agent[j],
        rrwm_results[(i, j)]["matches"],
    )
    print("evaluation:", rrwm_results[(i, j)]["evaluation"])

print_kwise_matches_with_true_ids(
    "k-wise MGM style RRWM matching: Agent 0 - 1 - 2",
    dets_0,
    dets_1,
    dets_2,
    kwise_matches,
)
print("evaluation:", kwise_eval)

## Comparison Table

In [ ]:
print_method_comparison_table(
    agent_pairs=agent_pairs,
    roco_results=roco_results,
    rrwm_results=rrwm_results,
    kwise_matches=kwise_matches,
    detections_by_agent=detections_by_agent,
)

## Matching Visualization

In [ ]:
for i, j in agent_pairs:
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        roco_results[(i, j)]["matches"],
        agent_i=i,
        agent_j=j,
        title=f"RoCo-style matching result: Agent {i} - Agent {j}",
        params=ROCO_PARAMS,
    )

In [ ]:
for i, j in agent_pairs:
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        rrwm_results[(i, j)]["matches"],
        agent_i=i,
        agent_j=j,
        title=f"Pairwise RRWM matching result: Agent {i} - Agent {j}",
        params=RRWM_PARAMS,
    )

In [ ]:
for i, j in agent_pairs:
    pairwise_from_kwise = kwise_to_pairwise_matches(
        kwise_matches,
        pair=(i, j),
    )
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        pairwise_from_kwise,
        agent_i=i,
        agent_j=j,
        title=f"k-wise MGM result shown as pairwise: Agent {i} - Agent {j}",
        params=KWISE_PARAMS,
    )